# Phase 4 — Posterior Predictive Inference

Companion notebook to `notes/phase4-predictive-inference.md`. We:

1. Compare the posterior on θ vs the posterior predictive on x̃ at
   mid-year (exercise 6 expected values).
2. Compute the year-end total predictive and P(over budget) using the
   correct quadratic-h variance, contrasted with the naïve iid formula.
3. Verify Monte Carlo posterior predictive sampling reproduces the
   closed-form variance — and breaks when correlation across periods
   is dropped.
4. Show the Gamma-Poisson predictive as a Negative Binomial.
5. Compute a Bayes factor between two Normal-Normal models with
   different priors.

In [ ]:
from __future__ import annotations

import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

from src.conjugate import NormalPosterior, GammaPosterior
from src.predictive import (
    posterior_predictive_normal,
    posterior_predictive_gamma_poisson,
    year_end_predictive_total,
    prob_over_budget,
    posterior_predictive_sample,
    log_marginal_likelihood_normal_normal,
    bayes_factor,
)

rng = np.random.default_rng(seed=20260701)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True})

## 1. Posterior vs predictive at mid-year (Exercise 6)

Posterior $\theta \mid x_{1:6} \sim N(1{,}085{,}000, 32{,}000^2)$,
$\sigma = 80{,}000$. The predictive for next month is
$\tilde x \mid x \sim N(\mu_n, \sigma_n^2 + \sigma^2)$.

In [ ]:
post = NormalPosterior(mu=1_085_000.0, sigma_sq=32_000.0**2)
sigma = 80_000.0

pred = posterior_predictive_normal(post, sigma_sq=sigma**2)
credible = post.credible_interval(0.95)
predictive_ci = (float(pred.ppf(0.025)), float(pred.ppf(0.975)))

print(f"Posterior:  μ={post.mean():,.0f}, σ={post.std():,.0f}")
print(f"Credible 95% CI:    [R$ {credible[0]:,.0f}, R$ {credible[1]:,.0f}]  width={credible[1]-credible[0]:,.0f}")
print(f"\nPredictive: μ={pred.mean():,.0f}, σ={pred.std():,.0f}")
print(f"Predictive 95% CI:  [R$ {predictive_ci[0]:,.0f}, R$ {predictive_ci[1]:,.0f}]  width={predictive_ci[1]-predictive_ci[0]:,.0f}")
print(f"\nWidth ratio (predictive / credible): {(predictive_ci[1]-predictive_ci[0]) / (credible[1]-credible[0]):.2f}×")

In [ ]:
grid = np.linspace(post.mean() - 4*pred.std(), post.mean() + 4*pred.std(), 700)
fig, ax = plt.subplots(figsize=(8, 4.4))
ax.plot(grid, stats.norm(post.mean(), post.std()).pdf(grid), color="steelblue", lw=2, label=f"Posterior θ  N({post.mean():,.0f}, {post.std():,.0f}²)")
ax.plot(grid, pred.pdf(grid), color="crimson", lw=2, label=f"Predictive x̃  N({pred.mean():,.0f}, {pred.std():,.0f}²)")
ax.axvspan(*credible, color="steelblue", alpha=0.12)
ax.axvspan(*predictive_ci, color="crimson", alpha=0.10)
ax.set_xlabel("value (R$)"); ax.set_ylabel("density")
ax.set_title("Posterior is for the parameter; predictive is for the future observation")
ax.ticklabel_format(style="plain", axis="x"); ax.legend(fontsize=9)
fig.tight_layout(); plt.show()

## 2. Year-end forecast: correct vs naïve variance (Exercise 7)

Mid-year posterior. $S_6 = 6{,}510{,}000$, budget $B = 13{,}200{,}000$,
horizon $h = 6$ months remaining.

In [ ]:
h = 6
S_obs = 6_510_000.0
B = 13_200_000.0

forecast = year_end_predictive_total(post, sigma_sq=sigma**2, n_remaining=h, observed_total=S_obs)
p_correct = forecast.prob_above(B)
ci_T = forecast.credible_interval(0.95)

naive_var = h * (post.variance() + sigma**2)  # WRONG formula — for comparison only
sigma_T_naive = np.sqrt(naive_var)
p_naive = float(stats.norm(forecast.mean, sigma_T_naive).sf(B))

print(f"Year-end total predictive:")
print(f"  E[T]                  = R$ {forecast.mean:,.0f}")
print(f"  σ_T (correct, h²σ²+hσ²) = R$ {forecast.std():,.0f}")
print(f"  σ_T (naïve, h(σ²+σ²)) = R$ {sigma_T_naive:,.0f}")
print(f"  95% credible interval = [R$ {ci_T[0]:,.0f}, R$ {ci_T[1]:,.0f}]")
print(f"\n  P(T > B = R$ {B:,.0f}):")
print(f"    correct = {p_correct:.4f}  ({p_correct:.1%})")
print(f"    naïve   = {p_naive:.4f}  ({p_naive:.1%})")
print(f"    diff    = {(p_correct-p_naive)*100:.1f} percentage points")

In [ ]:
grid_T = np.linspace(forecast.mean - 4*forecast.std(), forecast.mean + 4*forecast.std(), 700)
fig, ax = plt.subplots(figsize=(8, 4.4))
ax.plot(grid_T, stats.norm(forecast.mean, forecast.std()).pdf(grid_T), color="crimson", lw=2.5, label=f"Correct (h²σ²+hσ², σ_T={forecast.std()/1e3:,.0f}K)")
ax.plot(grid_T, stats.norm(forecast.mean, sigma_T_naive).pdf(grid_T), color="goldenrod", lw=1.5, ls="--", label=f"Naïve iid (σ_T={sigma_T_naive/1e3:,.0f}K)")
mask = grid_T >= B
ax.fill_between(grid_T[mask], stats.norm(forecast.mean, forecast.std()).pdf(grid_T[mask]), color="crimson", alpha=0.3, label=f"P(T>B)={p_correct:.1%}")
ax.axvline(B, color="black", ls="--", lw=1.5, label=f"Budget B = R$ {B:,.0f}")
ax.set_xlabel("annual total T (R$)"); ax.set_ylabel("density")
ax.set_title("Year-end total predictive — correct variance vs the naïve iid formula")
ax.ticklabel_format(style="plain", axis="x"); ax.legend(fontsize=9, loc="upper left")
fig.tight_layout(); plt.show()

## 3. Monte Carlo posterior predictive sampling

Reproduce the year-end total via simulation. The correct recipe shares
θ across all future months *within* a replication; the naïve `correlated=False`
draws each month independently and gives the wrong total variance.

In [ ]:
S = 100_000
samples_corr = posterior_predictive_sample(
    post, sigma_sq=sigma**2, n_samples=S, n_periods=h, seed=20260702
)
samples_naive = posterior_predictive_sample(
    post, sigma_sq=sigma**2, n_samples=S, n_periods=h, seed=20260702,
    correlated=False,
)

T_corr = S_obs + samples_corr.sum(axis=1)
T_naive = S_obs + samples_naive.sum(axis=1)

print(f"Closed-form correct σ_T: {forecast.std():,.0f}")
print(f"Monte Carlo correct σ_T:  {T_corr.std():,.0f}")
print(f"Monte Carlo naïve σ_T:    {T_naive.std():,.0f}")
print(f"\nMonte Carlo P(T>B) correct: {(T_corr > B).mean():.4f}")
print(f"Closed-form    P(T>B) correct: {p_correct:.4f}")
print(f"Monte Carlo    P(T>B) naïve:   {(T_naive > B).mean():.4f}")

## 4. Gamma-Poisson predictive (Negative Binomial)

Posterior $\lambda \mid x \sim \text{Gamma}(20, 7)$ from Phase 2
exercise 8. Predictive for next month's incident count is
$\text{NegBin}(20, 7/8)$.

In [ ]:
gp_post = GammaPosterior(alpha=20.0, beta=7.0)
gp_pred = posterior_predictive_gamma_poisson(gp_post)

print(f"Predictive mean:        {gp_pred.mean():.4f}  (= α/β = 20/7)")
print(f"Predictive variance:    {gp_pred.var():.4f}  (= α(β+1)/β² = 20·8/49)")
print(f"Poisson-only variance:  {gp_post.mean():.4f}  (would be E[λ]=20/7)")
print(f"Overdispersion factor:  {gp_pred.var() / gp_post.mean():.3f}")

k_grid = np.arange(0, 12)
pred_pmf = gp_pred.pmf(k_grid)
poisson_pmf = stats.poisson(mu=gp_post.mean()).pmf(k_grid)

fig, ax = plt.subplots(figsize=(7.5, 4))
width = 0.4
ax.bar(k_grid - width/2, pred_pmf, width=width, color="crimson", label=f"Predictive  NegBin(20, 7/8)  Var={gp_pred.var():.2f}")
ax.bar(k_grid + width/2, poisson_pmf, width=width, color="steelblue", label=f"Plug-in Poisson  λ̂={gp_post.mean():.3f}  Var={gp_post.mean():.2f}")
ax.set_xlabel("incident count k"); ax.set_ylabel("P(x̃ = k)")
ax.set_title("Gamma-Poisson predictive vs plug-in Poisson — overdispersion")
ax.legend(fontsize=9); fig.tight_layout(); plt.show()

## 5. Bayes factor between two priors

Same data, two priors that disagree on the prior mean. The closer
prior should have higher marginal likelihood; the Bayes factor
quantifies how much the data prefers it.

In [ ]:
true_mean = 1_080_000.0
data = rng.normal(loc=true_mean, scale=80_000.0, size=12)

log_close = log_marginal_likelihood_normal_normal(
    data=data, mu0=true_mean, sigma0_sq=150_000**2, sigma_sq=80_000**2
)
log_far = log_marginal_likelihood_normal_normal(
    data=data, mu0=true_mean + 5*150_000, sigma0_sq=150_000**2, sigma_sq=80_000**2
)

BF = bayes_factor(log_close, log_far)
log10_BF = (log_close - log_far) / np.log(10)

print(f"Log marginal — close prior (μ₀=true): {log_close:.3f}")
print(f"Log marginal — far prior (μ₀+5σ₀):    {log_far:.3f}")
print(f"\nBayes factor BF (close vs far): {BF:.3e}")
print(f"log10(BF):                    {log10_BF:.2f}")
print("\nJeffreys scale:")
print("  log10 BF ≥ 2  → decisive evidence for the close prior")
print("  log10 BF ≥ 1  → strong")
print("  log10 BF ≥ 0.5 → substantial")

---

**Next phase.** `notes/phase5-fyf-model.md` wires the conjugate
updaters (Phase 2), sequential engine (Phase 3), and predictive
machinery (this phase) into the complete annual FYF cycle and
simulates five scenarios, with diagnostics for model checking.